In [26]:
"""SIFT affine alignment with CLAHE and previous-section fallback.

Anchor Z0 stays fixed (default z530). Optionally register z529 to z530 first
so that z531 can try z529 if z530->z531 fails.

For each new section, try the last two successfully aligned reference sections,
newest first; for each reference try normal SIFT then CLAHE SIFT.
Empty sections are skipped. RANSAC/QC remain mandatory.
Outputs PNG previews, transforms to anchor, QC CSV, skipped sections.
Does NOT write a Corgie field.
"""
from pathlib import Path
from collections import deque
import csv

import cv2
import numpy as np
from cloudvolume import CloudVolume
from tqdm.auto import tqdm

# ============================================================
# 1. Configuration (source settings preserved from uploaded code)
# ============================================================
SRC = (
    "file:///volume_4/research/seongbong/align/"
    "cerebellum_align_verified2/img_masked"
)
OUT = Path(
    "/volume_4/research/seongbong/align/"
    "cerebellum_align_verified2/img_sift_new"
)
Z0, Z1, MIP = 325, 700, 8  # Z1 inclusive

# Make z529 available as fallback for z531 without changing anchor z530.
# Set to None to disable pre-anchor reference initialization.
PRE_ANCHOR_REF = None#Z0 - 1
REF_HISTORY = 2  # newest reference + one older reference
MAX_REF_GAP = 6  # do not match sections more than 3 z apart

N_FEATURES, RATIO, RANSAC_PX = 15000, 0.75, 5.0
MIN_MATCHES, MIN_INLIERS, MIN_INLIER_RATIO = 1, 10, 0.15
MIN_COVERAGE = 0.20
MAX_ROT_DEG, MIN_SCALE, MAX_SCALE = 12.0, 0.85, 1.15
CLAHE_CLIP, CLAHE_GRID = 2.0, (8, 8)

(OUT / "previews").mkdir(parents=True, exist_ok=True)

vol = CloudVolume(SRC, mip=MIP, progress=False, fill_missing=False)
b = vol.bounds
x0, x1 = int(b.minpt.x), int(b.maxpt.x)
y0, y1 = int(b.minpt.y), int(b.maxpt.y)
w, h = x1 - x0, y1 - y0
print("Bounds:", b, "width/height:", w, h)

if not (int(b.minpt.z) <= Z0 <= Z1 < int(b.maxpt.z)):
    raise ValueError("Requested Z range is outside volume bounds")


def read(z):
    a = np.asarray(vol[x0:x1, y0:y1, z:z + 1, :])
    if a.shape != (w, h, 1, 1):
        raise ValueError(f"Unexpected shape at z={z}: {a.shape}")
    return a[:, :, 0, 0].T.copy()  # OpenCV uses (Y, X)


def is_empty_section(img):
    return not np.any(np.isfinite(img) & (img != 0))


def u8(a):
    a = np.asarray(a, dtype=np.float32)
    valid = a[np.isfinite(a)]
    if valid.size == 0:
        raise ValueError("No finite pixels")
    lo, hi = np.percentile(valid, [1, 99])
    out = np.clip(
        (np.nan_to_num(a, nan=lo, posinf=hi, neginf=lo) - lo)
        * (255.0 / max(hi - lo, 1e-6)),
        0, 255
    )
    return out.astype(np.uint8)


def H(m):
    return np.vstack([m, [0.0, 0.0, 1.0]])


sift = cv2.SIFT_create(nfeatures=N_FEATURES, contrastThreshold=0.01)
matcher = cv2.BFMatcher(cv2.NORM_L2)
clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)


def features(a, use_clahe=False):
    img8 = u8(a)
    if use_clahe:
        img8 = clahe.apply(img8)
    return sift.detectAndCompute(img8, None)  # no SIFT spatial mask


def estimate(ref_features, mov_features, z, z_ref):
    kr, dr = ref_features
    km, dm = mov_features
    if dr is None or dm is None:
        raise RuntimeError(f"Z={z}: missing descriptors")

    fwd = matcher.knnMatch(dm, dr, k=2)
    rev = matcher.knnMatch(dr, dm, k=2)
    rev_ok = {
        m.queryIdx: m.trainIdx
        for pair in rev if len(pair) == 2
        for m, n in [pair]
        if m.distance < RATIO * n.distance
    }
    good = [
        m for pair in fwd if len(pair) == 2
        for m, n in [pair]
        if m.distance < RATIO * n.distance
        and rev_ok.get(m.trainIdx) == m.queryIdx
    ]
    if len(good) < MIN_MATCHES:
        raise RuntimeError(f"Z={z}: only {len(good)} mutual matches")

    pm = np.float32([km[m.queryIdx].pt for m in good])
    pr = np.float32([kr[m.trainIdx].pt for m in good])
    M, mask = cv2.estimateAffine2D(
        pm, pr, method=cv2.RANSAC,
        ransacReprojThreshold=RANSAC_PX,
        maxIters=20000, confidence=0.995, refineIters=20
    )
    if M is None or mask is None:
        raise RuntimeError(f"Z={z}: RANSAC failed")

    ok = mask.ravel().astype(bool)
    n_inliers, inlier_ratio = int(ok.sum()), float(ok.mean())
    if n_inliers < MIN_INLIERS or inlier_ratio < MIN_INLIER_RATIO:
        raise RuntimeError(f"Z={z}: weak inliers {n_inliers}/{len(good)}")

    cover_x = float(np.ptp(pm[ok, 0]) / w)
    cover_y = float(np.ptp(pm[ok, 1]) / h)
    if min(cover_x, cover_y) < MIN_COVERAGE:
        raise RuntimeError(
            f"Z={z}: low spatial coverage {cover_x:.2f}, {cover_y:.2f}"
        )

    scale = float(np.hypot(M[0, 0], M[1, 0]))
    rot = float(np.degrees(np.arctan2(M[1, 0], M[0, 0])))
    if not (MIN_SCALE <= scale <= MAX_SCALE and abs(rot) <= MAX_ROT_DEG):
        raise RuntimeError(
            f"Z={z}: implausible scale={scale:.3f}, rotation={rot:.2f}"
        )

    pred = cv2.transform(pm[ok][None], M)[0]
    residual = np.linalg.norm(pr[ok] - pred, axis=1)
    raw_disp = np.linalg.norm(pr[ok] - pm[ok], axis=1)
    center = np.array([w / 2.0, h / 2.0])
    cdisp = M[:, :2] @ center + M[:, 2] - center
    qc = dict(
        z_ref=z_ref, z_mov=z, gap=abs(z - z_ref),
        matches=len(good), inliers=n_inliers, inlier_ratio=inlier_ratio,
        coverage_x=cover_x, coverage_y=cover_y,
        rotation_deg=rot, scale=scale,
        center_dx=float(cdisp[0]), center_dy=float(cdisp[1]),
        median_before=float(np.median(raw_disp)),
        median_after=float(np.median(residual)),
        p95_after=float(np.percentile(residual, 95))
    )
    return M, qc


def entry(z, raw):
    return {"z": z, "raw": raw, "features": {}}


def get_features(e, method):
    if method not in e["features"]:
        e["features"][method] = features(
            e["raw"], use_clahe=(method == "clahe")
        )
    return e["features"][method]


def try_pair(ref, mov):
    """Try normal and CLAHE; return (P, qc, method), or error messages."""
    errors = []
    for method in ("normal", "clahe"):
        try:
            P, qc = estimate(
                get_features(ref, method),
                get_features(mov, method),
                z=mov["z"], z_ref=ref["z"]
            )
            return (P, qc, method), errors
        except RuntimeError as exc:
            errors.append(f"{ref['z']}->{mov['z']} {method}: {exc}")
    return None, errors


# ============================================================
# 6. Anchor and optional preceding reference
# ============================================================
raw_anchor = read(Z0)
if is_empty_section(raw_anchor):
    raise ValueError(f"Anchor Z={Z0} is empty")

G = {Z0: np.eye(3, dtype=np.float64)}
rows, skipped = [], []
anchor = entry(Z0, raw_anchor)
history = deque(maxlen=REF_HISTORY)

# Register the preceding section into the anchor frame, so z531 can use z529.
if PRE_ANCHOR_REF is not None:
    za = PRE_ANCHOR_REF
    if int(b.minpt.z) <= za < Z0:
        raw_pre = read(za)
        if not is_empty_section(raw_pre):
            pre = entry(za, raw_pre)
            result, errors = try_pair(anchor, pre)  # pre -> anchor
            if result is not None:
                P, qc, method = result
                G[za] = H(P)
                qc.update(status="pre_anchor", method=method)
                rows.append(qc)
                history.append(pre)
                print(f"Pre-anchor reference ready: z{za} -> z{Z0} ({method})")
            else:
                print("WARNING: Could not initialize preceding reference:")
                for err in errors:
                    print(" ", err)
        else:
            print(f"WARNING: Pre-anchor section z{za} is empty")

history.append(anchor)
cv2.imwrite(str(OUT / "previews" / f"z{Z0}.png"), u8(raw_anchor))

# ============================================================
# 7. Main alignment loop
# ============================================================
for z in tqdm(range(Z0 + 1, Z1 + 1), desc="SIFT + previous-ref fallback"):
    raw_cur = read(z)
    if is_empty_section(raw_cur):
        print(f"Z={z}: EMPTY -> skip")
    
        skipped.append(z)
    
        # Identity affine = zero displacement
        G[z] = np.eye(3, dtype=np.float64)
    
        rows.append(dict(
            z_ref=history[-1]["z"],
            z_mov=z,
            gap=z - history[-1]["z"],
            status="empty",
            method="identity"
        ))
    
        cv2.imwrite(
            str(OUT / "previews" / f"z{z}.png"),
            np.zeros((h, w), dtype=np.uint8)
        )
    
        continue

    cur = entry(z, raw_cur)
    selected = None
    failures = []
    for ref in reversed(history):  # latest successfully aligned reference first
        if z - ref["z"] > MAX_REF_GAP:
            continue
        result, errors = try_pair(ref, cur)
        failures.extend(errors)
        if result is not None:
            P, qc, method = result
            selected = (ref, P, qc, method)
            break

    if selected is None:
        for err in failures:
            print("FAILED:", err)
        raise RuntimeError(
            f"Z={z}: no valid SIFT affine from last {REF_HISTORY} references "
            f"within gap={MAX_REF_GAP}"
        )

    ref, P, qc, method = selected
    G[z] = G[ref["z"]] @ H(P)  # compose with the reference actually used
    qc.update(status="ok", method=method)
    rows.append(qc)

    aligned = cv2.warpAffine(
        raw_cur, G[z][:2].astype(np.float32), (w, h),
        flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0
    )
    cv2.imwrite(str(OUT / "previews" / f"z{z}.png"), u8(aligned))
    print(
        f"{ref['z']}->{z}: method={method}, gap={qc['gap']}, "
        f"rot={qc['rotation_deg']:+.2f}deg, scale={qc['scale']:.4f}, "
        f"after={qc['median_after']:.2f}px, "
        f"inliers={qc['inliers']}/{qc['matches']}"
    )
    history.append(cur)

# ============================================================
# 8. Save
# ============================================================
np.savez(
    OUT / f"affine_to_anchor{Z0}.npz",
    **{f"z{z}": g[:2] for z, g in G.items()}
)
fieldnames = list(dict.fromkeys(k for row in rows for k in row))
with (OUT / "pairwise_qc.csv").open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)
np.save(OUT / "skipped_sections.npy", np.asarray(skipped, dtype=np.int32))

print("\nCompleted")
print("Anchor:", Z0)
print("Valid transforms:", len(G))
print("Normal SIFT:", sum(r.get("method") == "normal" for r in rows))
print("CLAHE fallback:", sum(r.get("method") == "clahe" for r in rows))
print("Empty sections:", skipped)
print("Saved:", OUT)
print("IMPORTANT: QC values are for pairwise raw feature matches, not rendered neighbors.")


Bounds: Bbox([0, 0, 0],[975, 900, 1200], dtype=int32) width/height: 975 900


SIFT + previous-ref fallback:   0%|          | 0/375 [00:00<?, ?it/s]

325->326: method=normal, gap=1, rot=-0.00deg, scale=1.0003, after=0.31px, inliers=854/859
326->327: method=normal, gap=1, rot=+0.02deg, scale=1.0000, after=0.27px, inliers=701/708
327->328: method=normal, gap=1, rot=+0.02deg, scale=0.9999, after=0.28px, inliers=1079/1088
328->329: method=normal, gap=1, rot=-0.01deg, scale=1.0001, after=0.32px, inliers=798/808
329->330: method=normal, gap=1, rot=-0.00deg, scale=1.0000, after=0.26px, inliers=918/926
330->331: method=normal, gap=1, rot=-0.01deg, scale=1.0001, after=0.28px, inliers=853/866
331->332: method=normal, gap=1, rot=+0.02deg, scale=1.0001, after=0.30px, inliers=571/579
332->333: method=normal, gap=1, rot=+0.02deg, scale=1.0001, after=0.27px, inliers=961/971
333->334: method=normal, gap=1, rot=-3.06deg, scale=1.0042, after=0.37px, inliers=494/499
334->335: method=normal, gap=1, rot=+3.08deg, scale=0.9961, after=0.46px, inliers=369/378
335->336: method=normal, gap=1, rot=-0.01deg, scale=1.0000, after=0.33px, inliers=348/360
336->337

In [27]:
from copy import deepcopy

# --------------------------------------------------
# 1. Paths and settings
# --------------------------------------------------

MATRICES = OUT / "affine_to_anchor325.npz"

FIELD_TEMPLATE = (
    "file:///volume_4/research/seongbong/align/cerebellum_align_verified/"
    "align_sift/field/field_sift"
)

FIELD_OUT = (
    "file:///volume_4/research/seongbong/align/cerebellum_align_verified2/"
    "img_sift_new/field/field_sift"
)

TILE = 512

# --------------------------------------------------
# 2. Create CloudVolume field
# --------------------------------------------------

template = CloudVolume(
    FIELD_TEMPLATE,
    mip=MIP,
    progress=False
)

info = deepcopy(template.info)

if info["num_channels"] != 2:
    raise ValueError("Expected 2-channel displacement field")

if info["data_type"] != "float32":
    raise ValueError("Expected float32 displacement field")

out = CloudVolume(
    FIELD_OUT,
    mip=MIP,
    info=info,
    progress=False,
    compress=False
)

out.commit_info()

# Verify field coordinate bounds
fb = out.bounds

assert int(fb.minpt.x) == x0
assert int(fb.maxpt.x) == x1
assert int(fb.minpt.y) == y0
assert int(fb.maxpt.y) == y1

# --------------------------------------------------
# 3. Generate backward displacement field
# --------------------------------------------------

with np.load(MATRICES) as npz:

    for z in tqdm(range(Z0, Z1 + 1), desc="Writing fields"):

        key = f"z{z}"

        if key not in npz:
            raise KeyError(f"Missing affine: {key}")

        forward = np.asarray(
            npz[key],
            dtype=np.float64
        )

        if forward.shape != (2, 3):
            raise ValueError(
                f"{key}: unexpected shape {forward.shape}"
            )

        G_matrix = np.eye(3, dtype=np.float64)
        G_matrix[:2] = forward

        backward = np.linalg.inv(G_matrix)

        # Output pixel -> source pixel
        for ya in range(y0, y1, TILE):

            yb = min(ya + TILE, y1)

            for xa in range(x0, x1, TILE):

                xb = min(xa + TILE, x1)

                yy, xx = np.mgrid[ya:yb, xa:xb]

                local_x = xx.astype(np.float64) - x0
                local_y = yy.astype(np.float64) - y0

                src_x = (
                    backward[0, 0] * local_x
                    + backward[0, 1] * local_y
                    + backward[0, 2]
                )

                src_y = (
                    backward[1, 0] * local_x
                    + backward[1, 1] * local_y
                    + backward[1, 2]
                )

                dx = (src_x - local_x).astype(np.float32)
                dy = (src_y - local_y).astype(np.float32)

                # OpenCV: Y, X, C
                # CloudVolume: X, Y, Z, C
                tile = np.stack(
                    [dy.T, dx.T],
                    axis=-1
                )[:, :, None, :]

                out[
                    xa:xb,
                    ya:yb,
                    z:z+1,
                    :
                ] = tile

        center_x = (x1 - x0) / 2
        center_y = (y1 - y0) / 2

        center_dx = (
            (backward[0, 0] - 1) * center_x
            + backward[0, 1] * center_y
            + backward[0, 2]
        )

        center_dy = (
            backward[1, 0] * center_x
            + (backward[1, 1] - 1) * center_y
            + backward[1, 2]
        )

        print(
            f"Z={z}: "
            f"center displacement = "
            f"({center_dx:.2f}, {center_dy:.2f}) mip8 px"
        )

print("Completed:", FIELD_OUT)

Writing fields:   0%|          | 0/376 [00:00<?, ?it/s]

Z=325: center displacement = (0.00, 0.00) mip8 px
Z=326: center displacement = (0.02, -0.02) mip8 px
Z=327: center displacement = (-0.02, -0.01) mip8 px
Z=328: center displacement = (-0.02, 0.00) mip8 px
Z=329: center displacement = (-0.02, -0.00) mip8 px
Z=330: center displacement = (-0.01, -0.05) mip8 px
Z=331: center displacement = (-0.02, 0.02) mip8 px
Z=332: center displacement = (0.05, 0.05) mip8 px
Z=333: center displacement = (-0.03, 0.03) mip8 px
Z=334: center displacement = (45.22, 58.76) mip8 px
Z=335: center displacement = (-0.03, -0.08) mip8 px
Z=336: center displacement = (0.03, -0.09) mip8 px
Z=337: center displacement = (0.14, -0.03) mip8 px
Z=338: center displacement = (0.17, 0.04) mip8 px
Z=339: center displacement = (0.16, 0.02) mip8 px
Z=340: center displacement = (45.43, 58.93) mip8 px
Z=341: center displacement = (45.44, 58.95) mip8 px
Z=342: center displacement = (0.26, 0.23) mip8 px
Z=343: center displacement = (0.35, 0.12) mip8 px
Z=344: center displacement = (

In [ ]:
CUDA_VISIBLE_DEVICES=0 corgie render   --src_layer_spec '{"name":"img","type":"img","path":"file:///volume_4/research/seongbong/align/cerebellum_align_verified2/img_masked","args":{"data_mip":7}}'   --src_layer_spec '{"name":"field","type":"field","path":"file:///volume_4/research/seongbong/align/cerebellum_align_verified2/img_sift_new/field/field_sift","args":{"data_mip":8}}'   --dst_folder "file:///volume_4/research/seongbong/align/cerebellum_align_verified2/rendered_sift_new"   --start_coord "0,0,325"   --end_coord "249600,230400,336"   --coord_mip 0   --mip 8   --chunk_xy 512   --pad 128   --device cuda   --verbose